# Train CUT on horse2zebra (Google Colab)

Trains [Contrastive Unpaired Translation](https://github.com/taesungp/contrastive-unpaired-translation) on the `horse2zebra` dataset.

**Before running:** Runtime → Change runtime type → GPU (T4 is fine; A100/L4 faster).

The full paper config is 400 epochs (`n_epochs=200`, `n_epochs_decay=200`) which takes many hours on a T4. The cells below default to a much shorter demo run; change `N_EPOCHS` / `N_EPOCHS_DECAY` for a real training run.

## 1. Check GPU

In [1]:
!nvidia-smi

Fri Apr 24 13:02:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. (Optional) Mount Google Drive for persistent checkpoints

Colab VMs are ephemeral. Mounting Drive lets you resume training and keep results across sessions. Skip this cell if you don't care about persistence.

In [2]:
USE_DRIVE = True  # set True to persist checkpoints to Google Drive

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    CHECKPOINTS_DIR = '/content/drive/MyDrive/phd/models/cut_h2z_checkpoints'
    RESULTS_DIR = '/content/drive/MyDrive/phd/models/cut_h2z_results'
else:
    CHECKPOINTS_DIR = '/content/checkpoints'
    RESULTS_DIR = '/content/results'

import os
os.makedirs(CHECKPOINTS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print('checkpoints ->', CHECKPOINTS_DIR)
print('results     ->', RESULTS_DIR)

Mounted at /content/drive
checkpoints -> /content/drive/MyDrive/phd/models/cut_h2z_checkpoints
results     -> /content/drive/MyDrive/phd/models/cut_h2z_results


## 3. Clone the repo

In [3]:
%cd /content
![ -d contrastive-unpaired-translation ] || git clone https://github.com/valerybr/contrastive-unpaired-translation.git
%cd /content/contrastive-unpaired-translation

/content
Cloning into 'contrastive-unpaired-translation'...
remote: Enumerating objects: 294, done.
remote: Counting objects: 100% (152/152), done.
remote: Compressing objects: 100% (67/67), done.
remote: Total 294 (delta 96), reused 90 (delta 85), pack-reused 142 (from 1)
Receiving objects: 100% (294/294), 17.94 MiB | 20.80 MiB/s, done.
Resolving deltas: 100% (147/147), done.
/content/contrastive-unpaired-translation


## 4. Install dependencies

Colab already ships PyTorch + torchvision; we only need the repo's lightweight extras.

In [4]:
!pip install -q dominate visdom GPUtil packaging

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 38.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done


## 5. Patch a pre-existing argparse bug

Upstream `models/cut_model.py` declares `choices='(CUT, cut, FastCUT, fastcut)'` — a *string*, which argparse treats as a set of characters. Passing `--CUT_mode CUT` therefore fails as "invalid choice". The default works, but the launcher commands below pass the flag explicitly (and we may want FastCUT), so fix it.

In [5]:
import re, pathlib
p = pathlib.Path('models/cut_model.py')
src = p.read_text()
fixed = src.replace(
    "choices='(CUT, cut, FastCUT, fastcut)'",
    "choices=('CUT', 'cut', 'FastCUT', 'fastcut')",
)
if fixed != src:
    p.write_text(fixed)
    print('patched')
else:
    print('already patched or upstream changed')

patched


## 6. Download horse2zebra

~117 MB. Contains `trainA/` (horses), `trainB/` (zebras), `testA/`, `testB/`.

In [6]:
%%capture
!bash ./datasets/download_cut_dataset.sh horse2zebra
!ls datasets/horse2zebra && echo '---' && ls datasets/horse2zebra/trainA | head -3 && ls datasets/horse2zebra/trainA | wc -l

## 6b. Weights & Biases

Log losses every `print_freq` iters, and upload current image visuals every 5 epochs. Paste your wandb API key when prompted (find it at https://wandb.ai/authorize).

In [7]:
!pip install -q wandb
import wandb
from google.colab import userdata
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: valery-brodsky (valery-brodsky-ariel-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## 6c. wandb launcher

Writes a tiny wrapper that monkey-patches `util.visualizer.Visualizer` before `train.py` instantiates it:
- `print_current_losses` → mirror losses to `wandb.log` each time they're printed.
- `display_current_results` → upload the current model visuals as `wandb.Image` once per epoch when `epoch % WANDB_IMG_EVERY == 0` (default 5).

Runs `train.py` as `__main__` so the existing training loop is unchanged.

In [8]:
%%writefile wandb_train.py
import os
import runpy
import wandb
from util import visualizer as _v
from util import util as _u

IMG_EVERY = int(os.environ.get('WANDB_IMG_EVERY', '5'))

_orig_print = _v.Visualizer.print_current_losses
_orig_display = _v.Visualizer.display_current_results
_img_logged = set()


def print_current_losses(self, epoch, iters, losses, t_comp, t_data):
    payload = {f'loss/{k}': float(v) for k, v in losses.items()}
    payload.update({'epoch': epoch, 't_comp': t_comp, 't_data': t_data})
    wandb.log(payload)
    return _orig_print(self, epoch, iters, losses, t_comp, t_data)


def display_current_results(self, visuals, epoch, save_result):
    if epoch % IMG_EVERY == 0 and epoch not in _img_logged:
        _img_logged.add(epoch)
        imgs = {f'images/{label}': wandb.Image(_u.tensor2im(img))
                for label, img in visuals.items()}
        imgs['epoch'] = epoch
        wandb.log(imgs)
    return _orig_display(self, visuals, epoch, save_result)


_v.Visualizer.print_current_losses = print_current_losses
_v.Visualizer.display_current_results = display_current_results

wandb.init(project=os.environ.get('WANDB_PROJECT', 'cut'),
           name=os.environ.get('WANDB_NAME'))
try:
    runpy.run_path('train.py', run_name='__main__')
finally:
    wandb.finish()

Writing wandb_train.py


## 7. Train

Key flags:
- `--CUT_mode CUT` — full CUT (identity NCE, `lambda_NCE=1.0`). Use `FastCUT` for the faster variant.
- `--display_id -1` — disable visdom (Colab can't easily host the visdom server).
- `--num_threads 2` — Colab free tiers have 2 vCPUs.
- `--save_epoch_freq` — how often checkpoints are written (every N epochs).
- `--n_epochs` / `--n_epochs_decay` — constant-LR epochs, then linearly-decayed epochs. Paper: 200 + 200.

**Resume** a previous run by adding `--continue_train --epoch_count <next_epoch>`.

In [ ]:
EXP_NAME = 'horse2zebra_CUT'
N_EPOCHS = 5          # constant-LR epochs (paper: 200)
N_EPOCHS_DECAY = 5    # decay epochs         (paper: 200)
SAVE_EPOCH_FREQ = 2
WANDB_PROJECT = 'cut'
WANDB_IMG_EVERY = 5   # upload model visuals to wandb every N epochs

cmd = f"""WANDB_PROJECT={WANDB_PROJECT} WANDB_NAME={EXP_NAME} WANDB_IMG_EVERY={WANDB_IMG_EVERY} \
python wandb_train.py \
  --dataroot ./datasets/horse2zebra \
  --name {EXP_NAME} \
  --CUT_mode CUT \
  --checkpoints_dir {CHECKPOINTS_DIR} \
  --n_epochs {N_EPOCHS} \
  --n_epochs_decay {N_EPOCHS_DECAY} \
  --save_epoch_freq {SAVE_EPOCH_FREQ} \
  --display_id -1 \
  --num_threads 2"""
print(cmd)
!{cmd}

WANDB_PROJECT=cut WANDB_NAME=horse2zebra_CUT WANDB_IMG_EVERY=5 python wandb_train.py   --dataroot ./datasets/horse2zebra   --name horse2zebra_CUT   --CUT_mode CUT   --checkpoints_dir /content/drive/MyDrive/phd/models/cut_h2z_checkpoints   --n_epochs 5   --n_epochs_decay 5   --save_epoch_freq 2   --display_id -1   --num_threads 2
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: valery-brodsky (valery-brodsky-ariel-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
]11;?]11;?wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ setting up run vg84ikio (0.2s)
wandb: ⣾ setting up run vg84ikio (0.2s)
wandb: ⣷ setting up run vg84ikio (0.2s)
wandb: ⣯ setting up run vg84ikio (0.2s)
wandb: ⣟ setting up run vg84ikio (0.2s)
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /content/contrastive-unpaired-translation/wandb/run-20260424_1

## 8. Inference on the test set

`test.py` loads the generator from `<checkpoints_dir>/<name>/latest_net_G.pth` and writes an HTML gallery to `<results_dir>/<name>/test_latest/index.html`.

In [ ]:
cmd = f"""python test.py \
  --dataroot ./datasets/horse2zebra \
  --name {EXP_NAME} \
  --CUT_mode CUT \
  --checkpoints_dir {CHECKPOINTS_DIR} \
  --results_dir {RESULTS_DIR} \
  --phase test \
  --num_test 20"""
print(cmd)
!{cmd}

## 9. Preview generated images

In [ ]:
import glob, os
from IPython.display import Image, display, HTML

img_dir = os.path.join(RESULTS_DIR, EXP_NAME, 'test_latest', 'images')
reals = sorted(glob.glob(os.path.join(img_dir, '*_real_A.png')))[:6]
for real in reals:
    fake = real.replace('_real_A.png', '_fake_B.png')
    display(HTML(f"<h4>{os.path.basename(real).replace('_real_A.png','')}</h4>"))
    display(Image(real, width=256), Image(fake, width=256))

## 10. (Optional) Train FastCUT instead

FastCUT skips the identity NCE, uses `lambda_NCE=10.0` and flip-equivariance. Lighter and ~2× faster than CUT.

Its `modify_commandline_options` overrides `n_epochs=150, n_epochs_decay=50` by default — override on the CLI to shorten for a demo.

In [ ]:
# Uncomment to run
# !python train.py \
#   --dataroot ./datasets/horse2zebra \
#   --name horse2zebra_FastCUT \
#   --CUT_mode FastCUT \
#   --checkpoints_dir {CHECKPOINTS_DIR} \
#   --n_epochs 5 --n_epochs_decay 5 \
#   --display_id -1 --num_threads 2